<a href="https://colab.research.google.com/github/cz2112/hw/blob/main/Notebooks/Chap13/13_4_Graph_Attention_Networks.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Notebook 13.4: Graph attention networks**

This notebook builds a graph attention mechanism from scratch, as discussed in section 13.8.6 of the book and illustrated in figure 13.12c

Work through the cells below, running each cell in turn. In various places you will see the words "TODO". Follow the instructions at these places and make predictions about what is going to happen or write code to complete the functions.

Contact me at udlbookmail@gmail.com if you find any mistakes or have any suggestions.



In [1]:
import numpy as np
import matplotlib.pyplot as plt

The self-attention mechanism maps $N$ inputs $\mathbf{x}_{n}\in\mathbb{R}^{D}$ and returns $N$ outputs $\mathbf{x}'_{n}\in \mathbb{R}^{D}$.  



In [2]:
# Set seed so we get the same random numbers
np.random.seed(1)
# Number of nodes in the graph
N = 8
# Number of dimensions of each input
D = 4

# Define a graph
A = np.array([[0,1,0,1,0,0,0,0],
              [1,0,1,1,1,0,0,0],
              [0,1,0,0,1,0,0,0],
              [1,1,0,0,1,0,0,0],
              [0,1,1,1,0,1,0,1],
              [0,0,0,0,1,0,1,1],
              [0,0,0,0,0,1,0,0],
              [0,0,0,0,1,1,0,0]]);
print(A)

# Let's also define some random data
X = np.random.normal(size=(D,N))

[[0 1 0 1 0 0 0 0]
 [1 0 1 1 1 0 0 0]
 [0 1 0 0 1 0 0 0]
 [1 1 0 0 1 0 0 0]
 [0 1 1 1 0 1 0 1]
 [0 0 0 0 1 0 1 1]
 [0 0 0 0 0 1 0 0]
 [0 0 0 0 1 1 0 0]]


We'll also need the weights and biases for the keys, queries, and values (equations 12.2 and 12.4)

In [3]:
# Choose random values for the parameters
omega = np.random.normal(size=(D,D))
beta = np.random.normal(size=(D,1))
phi = np.random.normal(size=(2*D,1))

We'll need a softmax operation that operates on the columns of the matrix and a ReLU function as well

In [4]:
# Define softmax operation that works independently on each column
def softmax_cols(data_in):
  # Exponentiate all of the values
  exp_values = np.exp(data_in) ;
  # Sum over columns
  denom = np.sum(exp_values, axis = 0);
  # Replicate denominator to N rows
  denom = np.matmul(np.ones((data_in.shape[0],1)), denom[np.newaxis,:])
  # Compute softmax
  softmax = exp_values / denom
  # return the answer
  return softmax


# Define the Rectified Linear Unit (ReLU) function
def ReLU(preactivation):
  activation = preactivation.clip(0.0)
  return activation


In [5]:
# Now let's compute self attention in matrix form
def graph_attention(X, omega, beta, phi, A):

    # 1. Compute X_prime
    # X_prime shape: (D, N)
    X_prime = omega @ X + beta

    # 2. Compute S
    # We need to compute S_{ij} = phi^T * [X'_i || X'_j] for all pairs.
    # To do this using matrices, we can use broadcasting.
    # Let's break phi into two halves: phi_1 and phi_2, each of shape (D, 1)
    D = X.shape[0]
    phi_1 = phi[:D, :] # Weights for the "target" node i
    phi_2 = phi[D:, :] # Weights for the "source" node j

    # Compute the partial scores:
    # term1 shape: (1, N) -> contribution of node i
    term1 = phi_1.T @ X_prime
    # term2 shape: (1, N) -> contribution of node j
    term2 = phi_2.T @ X_prime

    # Combine them to form the N x N score matrix S.
    # We want S[i, j] = term1[0, i] + term2[0, j]
    # Adding a column vector (N, 1) and a row vector (1, N) broadcasts to (N, N)
    S = term1.T + term2 # Shape: (N, N)

    # 3. Apply the mask
    # A node should only attend to its neighbors and itself.
    # Create the mask A + I
    mask = A + np.eye(A.shape[0])

    # Set S to a very large negative number everywhere the mask is zero
    S = np.where(mask == 0, -1e20, S)

    # 4. Run the softmax function to compute the attention values
    # We softmax over the columns (source nodes j for a given target node i)
    # The output S_softmax has shape (N, N)
    attention_weights = softmax_cols(S)

    # 5. Postmultiply X' by the attention values
    # X_prime is (D, N). attention_weights is (N, N).
    # Result is (D, N).
    # This computes sum_j alpha_ij * X'_j for each target node i.
    aggregated_features = X_prime @ attention_weights.T

    # 6. Apply the ReLU function
    output = ReLU(aggregated_features)

    return output

In [6]:
# Test out the graph attention mechanism
np.set_printoptions(precision=3)
output = graph_attention(X, omega, beta, phi, A);
print("Correct answer is:")
print("[[0.    0.028 0.37  0.    0.97  0.    0.    0.698]")
print(" [0.    0.    0.    0.    1.184 0.    2.654 0.  ]")
print(" [1.13  0.564 0.    1.298 0.268 0.    0.    0.779]")
print(" [0.825 0.    0.    1.175 0.    0.    0.    0.  ]]]")


print("Your answer is:")
print(output)

Correct answer is:
[[0.    0.028 0.37  0.    0.97  0.    0.    0.698]
 [0.    0.    0.    0.    1.184 0.    2.654 0.  ]
 [1.13  0.564 0.    1.298 0.268 0.    0.    0.779]
 [0.825 0.    0.    1.175 0.    0.    0.    0.  ]]]
Your answer is:
[[2.032e+00 3.844e-03 0.000e+00 1.396e-02 1.197e+00 9.634e-02 0.000e+00
  1.900e-01]
 [1.283e-01 0.000e+00 0.000e+00 0.000e+00 1.514e+00 6.566e-02 4.916e+00
  1.135e-01]
 [1.746e+00 0.000e+00 0.000e+00 1.682e-02 2.130e-01 4.269e-02 0.000e+00
  1.002e-01]
 [0.000e+00 0.000e+00 0.000e+00 0.000e+00 0.000e+00 0.000e+00 0.000e+00
  0.000e+00]]


TODO -- Try to construct a dot-product self-attention mechanism as in practical 12.1 that respects the geometry of the graph and has zero attention between non-neighboring nodes by combining figures 13.12a and 13.12b.


In [7]:
# TODO -- Try to construct a dot-product self-attention mechanism as in practical 12.1
# that respects the geometry of the graph and has zero attention between non-neighboring nodes
# by combining figures 13.12a and 13.12b.

def masked_dot_product_attention(X, W_q, W_k, W_v, A):
    # 1. Compute Queries, Keys, and Values
    # Shapes: (D, N)
    Q = W_q @ X
    K = W_k @ X
    V = W_v @ X

    # 2. Compute dot-product attention scores S = Q^T * K
    # Shape: (N, N)
    S = Q.T @ K

    # 3. Apply the graph mask (A + I)
    mask = A + np.eye(A.shape[0])
    # Set non-neighbors to -inf so softmax makes them 0
    S = np.where(mask == 0, -1e20, S)

    # 4. Softmax over columns
    attention_weights = softmax_cols(S)

    # 5. Multiply Values by attention weights
    # V is (D, N), attention_weights is (N, N).
    # To sum over the source nodes (columns of V) using the weights for target node i (columns of attention_weights),
    # we do V @ attention_weights
    output = V @ attention_weights

    return output

# Test it with random weights
np.random.seed(42)
W_q = np.random.normal(size=(D,D))
W_k = np.random.normal(size=(D,D))
W_v = np.random.normal(size=(D,D))

dot_output = masked_dot_product_attention(X, W_q, W_k, W_v, A)
print("\nMasked Dot-Product Attention Output:")
print(dot_output)


Masked Dot-Product Attention Output:
[[ 3.482 -1.587 -0.893 -1.596  0.736 -0.249  0.729  0.775]
 [ 2.713  0.006 -1.182  0.124 -1.125  2.222 -1.157  0.371]
 [-0.859  0.96   0.141  1.001 -2.023  0.792 -2.034 -0.931]
 [ 1.71  -1.552 -0.434 -1.596  3.645 -0.824  3.66   1.856]]
